In [ ]:
!pip freeze > requirements-lock.txt

In [ ]:
# Cell 1: Dependencies & Repositories
!pip install -q edge-tts pydub

# Install openWakeWord from source
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword --no-deps

# Clone Piper generator for base phoneme generation
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!cd piper-sample-generator && git checkout 213d4d5

# Audio processing & training tools
!pip install -q piper-tts piper-phonemize-cross webrtcvad
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14
!pip install -q audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6
!pip install -q datasets==2.14.6 deep-phonemizer==0.0.19 pronouncing==0.2.0

# Export & conversion tools
!pip install -q onnxruntime==1.22.1 ai_edge_litert==1.4.0 onnxsim onnx2tf onnx==1.19.1 onnx_graphsurgeon sng4onnx

# Force numpy < 2.0 to maintain PyTorch/TensorFlow C-API compatibility
!pip install -q "numpy<2.0" --force-reinstall


In [ ]:
# 1. Install missing runtime packages
!pip install -q speexdsp-ns ai-edge-litert

# 2. Re-pin NumPy to <2.0 (critical for openWakeWord dataset reading)
!pip install -q "numpy<2.0" --force-reinstall

# 3. Test verification
import openwakeword
import edge_tts
import piper_phonemize
import ai_edge_litert
import numpy as np

print(f"NumPy version: {np.__version__}")
print("✅ All dependencies verified and ready for Cell 2!")


In [ ]:
# since i was getting the numpy clash error, du  to old numpy loaded in ram, i am forcing the session to restart by killing the os, and collab auto starts session and connects to t4 correctly.import os
os._exit(0)


In [ ]:
#now verify that numpy is correctly loaded with correct version.
import numpy as np
print(f"Active NumPy: {np.__version__}")


Create a new cell, run this, and let it restart the runtime so the updated C-extensions reload cleanly into memory:

In [ ]:
# Upgrade datasets and pyarrow to modern compatible builds while locking numpy<2.0
!pip install -q -U datasets pyarrow "numpy<2.0"

import os
os._exit(0)


In [ ]:
# Cell 2: Base Datasets & Feature Backbones (100% Reliable Version)
import os
import sys
import glob
import shutil
from pathlib import Path
import datasets
import numpy as np
import scipy.io.wavfile
from huggingface_hub import snapshot_download
from tqdm import tqdm

# 1. Models (Embedding & Spectrogram)
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q -c https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q -c https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q -c https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q -c https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# 2. MIT Room Impulse Responses (Checks if already downloaded)
output_dir_rir = "./mit_rirs"
existing_rirs = glob.glob(f"{output_dir_rir}/*.wav")
if len(existing_rirs) >= 100:
    print(f"✅ MIT RIRs already intact ({len(existing_rirs)} files). Skipping.")
else:
    os.makedirs(output_dir_rir, exist_ok=True)
    print("Downloading MIT RIR audio files...")
    rir_repo_path = snapshot_download(
        repo_id="davidscripka/MIT_environmental_impulse_responses",
        repo_type="dataset"
    )
    for wav_file in glob.glob(f"{rir_repo_path}/16khz/*.wav"):
        shutil.copy(wav_file, output_dir_rir)
    print(f"✅ Extracted {len(glob.glob(f'{output_dir_rir}/*.wav'))} RIR impulse response files.")

# 3. Background Noises (Populates audioset_16k and fma using ESC-50)
dir_audioset = "./audioset_16k"
dir_fma = "./fma"
os.makedirs(dir_audioset, exist_ok=True)
os.makedirs(dir_fma, exist_ok=True)

if len(glob.glob(f"{dir_audioset}/*.wav")) < 200 or len(glob.glob(f"{dir_fma}/*.wav")) < 200:
    print("Downloading environmental & background sounds (ESC-50)...")
    esc_dataset = datasets.load_dataset("ashraq/esc50", split="train")
    esc_dataset = esc_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))

    print("Processing background audio into training folders...")
    for idx, row in enumerate(tqdm(esc_dataset, total=600, desc="Saving Background Audio")):
        if idx < 300:
            target_path = os.path.join(dir_audioset, f"bg_noise_{idx}.wav")
        elif idx < 600:
            target_path = os.path.join(dir_fma, f"bg_music_noise_{idx}.wav")
        else:
            break
        scipy.io.wavfile.write(target_path, 16000, (row['audio']['array'] * 32767).astype(np.int16))

    print(f"✅ Generated {len(glob.glob(f'{dir_audioset}/*.wav'))} noise clips in {dir_audioset}.")
    print(f"✅ Generated {len(glob.glob(f'{dir_fma}/*.wav'))} background clips in {dir_fma}.")
else:
    print("✅ Background noise directories already populated. Skipping.")

# 4. Precomputed Negative Background Features (ACAV100M & Validation)
if not os.path.exists("./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    print("Downloading ACAV100M negative features (~3.5GB, takes ~2-3 mins)...")
    !wget -q -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

if not os.path.exists("validation_set_features.npy"):
    print("Downloading validation negative features...")
    !wget -q -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

print("\n🎉 Cell 2 Complete! All background datasets, models, and backbones are ready.")


In [ ]:
# Cell 3: Audio Demo Player
import edge_tts
from IPython.display import Audio, display

demo_configs = [
    ("hi-IN-MadhurNeural", "वी-सार", "Hindi Male (Madhur) - वी-सार"),
    ("hi-IN-SwaraNeural", "वी-सार", "Hindi Female (Swara) - वी-सार"),
    ("en-IN-PrabhatNeural", "Viee-saar", "Indian English Male (Prabhat) - Vee-saar"),
    ("en-IN-NeerjaNeural", "Viee-saar", "Indian English Female (Neerja) - Vee-saar"),
    ("en-IN-PrabhatNeural", "Viisaar", "Indian English Male (Prabhat) - Viisaar"),
    ("en-IN-NeerjaNeural", "Viisaar", "Indian English Female (Neerja) - Viisaar"),
    ("en-IN-PrabhatNeural", "Vee-saar", "Indian English Male (Prabhat) - Vee-saar"),
    ("en-IN-NeerjaNeural", "Vee-saar", "Indian English Female (Neerja) - Vee-saar"),
    ("en-IN-PrabhatNeural", "Viisaar", "Indian English Male (Prabhat) - Viisaar"),
]

print("Generating demo audio clips...")
for idx, (voice, text, label) in enumerate(demo_configs):
  temp_file = f"demo_audio_{idx}.mp3"
  communicate = edge_tts.Communicate(text=text, voice=voice)
  await communicate.save(temp_file)
  print(f"\n▶ {label}")
  display(Audio(temp_file, autoplay=False))


In [ ]:
# Cell 4: Config & Multi-Voice Dataset Generation
import asyncio
import glob
import itertools
import os
import shutil
import sys
import edge_tts
from pydub import AudioSegment
import yaml

# 1. Write Training Configuration
config = yaml.load(
    open("openwakeword/examples/custom_model.yml", "r").read(), yaml.Loader
)
config["target_phrase"] = ["vee_saar"]
config["model_name"] = "visar_edge"
config["n_samples"] = 5000  # Base synthetic samples
config["n_samples_val"] = 500
config["steps"] = 35000  # Training steps
config["target_accuracy"] = 0.5
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"
config["max_negative_weight"] = 3000
config["background_paths"] = ["./audioset_16k", "./fma"]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {
    "ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
}

with open("my_model.yaml", "w") as f:
  yaml.dump(config, f)

# 2. Run Piper Base Clip Generation
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

# 3. Inject Native Hindi & Indian English Neural Clips
target_dir = "my_custom_model/positive/"
os.makedirs(target_dir, exist_ok=True)

TEXT_CONFIGS = [
    ("hi-IN-MadhurNeural", "वी-सार"),
    ("hi-IN-MadhurNeural", "वीसार"),
    ("hi-IN-MadhurNeural", "वी सार"),

    ("hi-IN-SwaraNeural", "वी-सार"),
    ("hi-IN-SwaraNeural", "वीसार"),
    ("hi-IN-SwaraNeural", "वी सार"),

    ("en-IN-PrabhatNeural", "Vee-saar"),
    ("en-IN-PrabhatNeural", "Viisaar"),
    ("en-IN-PrabhatNeural", "Viee-saar"),

    ("en-IN-NeerjaNeural", "Vee-saar"),
    ("en-IN-NeerjaNeural", "Viisaar"),
    ("en-IN-NeerjaNeural", "Viee-saar"),
]
RATES = ["-15%", "-5%", "+0%", "+5%", "+15%"]
PITCHES = ["-15Hz", "-5Hz", "+0Hz", "+5Hz", "+15Hz"]


async def generate_and_convert(voice, text, rate, pitch, idx):
  mp3_path = f"/tmp/edge_in_{idx}.mp3"
  wav_path = f"{target_dir}/edge_in_{idx}.wav"
  communicate = edge_tts.Communicate(
      text=text, voice=voice, rate=rate, pitch=pitch
  )
  await communicate.save(mp3_path)

  # Standardize to 16kHz, 1-channel, 16-bit PCM WAV
  sound = AudioSegment.from_mp3(mp3_path)
  sound = (
      sound.set_frame_rate(16000)
      .set_channels(1)
      .set_sample_width(2)
  )
  sound.export(wav_path, format="wav")
  if os.path.exists(mp3_path):
    os.remove(mp3_path)


async def run_indian_synthesis():
  tasks = []
  count = 0
  for voice, text in TEXT_CONFIGS:
    for rate, pitch in itertools.product(RATES, PITCHES):
      tasks.append(generate_and_convert(voice, text, rate, pitch, count))
      count += 1
  for i in range(0, len(tasks), 20):
    await asyncio.gather(*tasks[i : i + 20])
  print(f"✅ Injected {count} Indian Edge-TTS positive samples.")


await run_indian_synthesis()

# 4. Inject Real Microphone Recordings (if folder exists)
real_clips = glob.glob("my_real_voice/*.wav")
for i, clip in enumerate(real_clips):
  shutil.copy(clip, f"{target_dir}/real_voice_devansh_{i}.wav")
print(
    f"✅ Injected {len(real_clips)} user microphone samples from"
    " 'my_real_voice/'."
)


In [ ]:
!python -c 'import torchaudio, os; p = os.path.dirname(torchaudio.__file__) + "/__init__.py"; open(p, "a").write("\ndef set_audio_backend(backend):\n    pass\n")'
print("✅ Torchaudio backend compatibility patch applied successfully.")


In [ ]:
import glob
import os
from pydub import AudioSegment

input_files = glob.glob("my_real_voice/*")
print(f"Found {len(input_files)} user audio recordings.")

for idx, file_path in enumerate(input_files):
  ext = os.path.splitext(file_path)[1].lower()
  if ext in [".wav", ".mp3", ".m4a", ".ogg", ".flac"]:
    # Load and normalize audio
    sound = AudioSegment.from_file(file_path)
    sound = (
        sound.set_frame_rate(16000)
        .set_channels(1)
        .set_sample_width(2)
    )

    clean_target = f"my_real_voice/normalized_voice_{idx}.wav"
    sound.export(clean_target, format="wav")

print("✅ All user voice samples normalized to 16kHz Mono WAV.")


In [ ]:
!pip install -q webrtcvad-wheels
# Fallback build just in case:
!pip install -q webrtcvad || true

import webrtcvad
print("✅ webrtcvad successfully installed and imported!")


In [ ]:
# Patch Piper's torch.load to permit loading the legacy checkpoint architecture
file_path = "/content/piper-sample-generator/generate_samples.py"
with open(file_path, "r") as f:
    content = f.read()

# Replace strict load with weights_only=False
patched = content.replace("torch.load(model_path)", "torch.load(model_path, weights_only=False)")
with open(file_path, "w") as f:
    f.write(patched)

print("✅ Patched Piper generator for modern PyTorch!")


RERUN CELL 4 again now. before proceeding.

In [ ]:
# Patch openWakeWord data generator to bypass dead AWS S3 bucket
patch_code = """

# --- OFFLINE ADVERSARIAL PHONEME GENERATOR ---
import random
import pronouncing

def generate_adversarial_texts(input_text, N=1000, include_partial_phrase=1.0, include_input_words=0.2, phonemizer_mdl_path=None):
    parts = input_text.replace("_", " ").split()

    # Syllables + phonetically dangerous English & Indian words
    hard_negatives = [
        "visa", "visas", "visor", "visors", "caesar", "laser", "teaser", "geyser",
        "freezer", "vicar", "weesar", "vector", "vivid", "vision", "visit", "reset",
        "piece of", "feature", "future", "vessel", "verser", "blazer", "closer",
        "scissor", "sensor", "mixer", "fixer",
        # Indian/Hindi close-sounding phonemes
        "vichar", "vishal", "vikas", "bazaar", "deewar", "sarkar", "sansar",
        "virodh", "vidya", "viman", "videsh", "vibhag", "vishesh", "vidhan",
        "namaste", "accha", "theek", "shri"
    ]
    for p in parts:
        hard_negatives.append(p)

    # Pull rhyming sounds via local CMU dict
    rhyme_pool = []
    try:
        for p in parts:
            clean = "".join([c for c in p if c.isalpha()])
            r = pronouncing.rhymes(clean)
            if r:
                rhyme_pool.extend(r[:25])
    except Exception:
        pass

    if not rhyme_pool:
        rhyme_pool = ["see", "tea", "bee", "key", "car", "far", "tar", "bar", "star"]

    ambient_words = [
        "the", "audio", "system", "computer", "device", "light", "turn", "off",
        "play", "music", "stop", "cancel", "volume", "louder", "quieter", "hello"
    ]

    candidates = list(set(hard_negatives + rhyme_pool + ambient_words))
    results = []
    while len(results) < N:
        w1 = random.choice(candidates)
        if random.random() < 0.35:
            w2 = random.choice(candidates)
            results.append(f"{w1} {w2}")
        else:
            results.append(w1)

    return results[:N]
# --- END OFFLINE PATCH ---
"""

with open("/content/openwakeword/openwakeword/data.py", "a") as f:
    f.write(patch_code)

# Test adversarial generator
from openwakeword.data import generate_adversarial_texts
test_words = generate_adversarial_texts("vee_saar", N=5)
print(f"✅ Patch verified! Test adversarial phrases: {test_words}")


In [ ]:
import sys
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips


now lets move to cell 5 things.

In [ ]:
# Shim torchaudio.info and torchaudio.load with soundfile
patch_script = """
import soundfile as sf
import torch
from collections import namedtuple

AudioMetaData = namedtuple(
    "AudioMetaData",
    ["sample_rate", "num_frames", "num_channels", "bits_per_sample", "encoding"]
)

def _shim_info(file_path, format=None):
    s = sf.info(str(file_path))
    return AudioMetaData(
        sample_rate=s.samplerate,
        num_frames=s.frames,
        num_channels=s.channels,
        bits_per_sample=16,
        encoding="PCM_S"
    )

def _shim_load(file_path, frame_offset=0, num_frames=-1, normalize=True, channels_first=True, format=None):
    start = frame_offset
    stop = None if num_frames in (-1, None) else frame_offset + num_frames
    data, sr = sf.read(str(file_path), start=start, stop=stop, dtype="float32", always_2d=True)
    tensor = torch.from_numpy(data)
    if channels_first:
        tensor = tensor.t()
    return tensor, sr

info = _shim_info
load = _shim_load
"""

import torchaudio, os
init_path = os.path.dirname(torchaudio.__file__) + "/__init__.py"
with open(init_path, "a") as f:
    f.write(patch_script)

# Verify the patch works in Python
!python -c 'import torchaudio,os; meta = torchaudio.info("mit_rirs/" + os.listdir("mit_rirs")[0]); print(f"✅ Torchaudio patch verified! Sample rate: {meta.sample_rate}Hz, Frames: {meta.num_frames}")'


In [ ]:
import glob
import os

# Recursively locate and remove generated feature files
found = False
for f in glob.glob("my_custom_model/**/*.npy", recursive=True):
  if "ACAV100M" not in f and "validation_set" not in f:
    os.remove(f)
    print(f"🗑️ Removed: {f}")
    found = True

# Also check root of my_custom_model
for f in glob.glob("my_custom_model/*.npy"):
  if "ACAV100M" not in f and "validation_set" not in f:
    os.remove(f)
    print(f"🗑️ Removed: {f}")
    found = True

if not found:
  print("No .npy files found in my_custom_model. Checking full tree...")
  for f in glob.glob("**/*features_train*.npy", recursive=True):
    os.remove(f)
    print(f"🗑️ Removed: {f}")

print("✅ Stale feature files cleared.")


In [ ]:
# Cell 5: Audio Augmentation
import sys
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

print("✅ Acoustic augmentation complete.")


In [ ]:
# Cell 6: Train the Model
import sys

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model
print("✅ Training complete!")


In [ ]:
!pip install -q onnxscript
print("✅ onnxscript installed.")


In [ ]:
import glob
import os
import torch
import onnxscript

print("Checking for existing model checkpoints...")
checkpoints = glob.glob("my_custom_model/**/*.pt", recursive=True) + glob.glob("my_custom_model/*.pt")
print(f"Found checkpoints: {checkpoints}")

if checkpoints:
    print("Found checkpoint! Exporting directly to ONNX...")
    # Load the latest checkpoint
    latest_pt = checkpoints[-1]
    model = torch.load(latest_pt, map_location="cpu", weights_only=False)
    if hasattr(model, "eval"):
        model.eval()

    # openWakeWord feature extractor input shape: [batch, 16, 96]
    dummy_input = torch.rand(1, 16, 96)
    onnx_out = "my_custom_model/visar_edge.onnx"
    torch.onnx.export(
        model,
        dummy_input,
        onnx_out,
        opset_version=13,
        input_names=["input"],
        output_names=["output"]
    )
    print(f"✅ Successfully exported: {onnx_out}")
else:
    print("No raw .pt file saved directly. Proceed to Step 3.")


In [ ]:
import os
import sys

# 1. Patch train.py to save raw PyTorch model (.pt) before ONNX export
train_script = "openwakeword/openwakeword/train.py"
with open(train_script, "r") as f:
  code = f.read()

target = 'oww.export_model(model=best_model'
replacement = """# Save raw PyTorch checkpoint for future fine-tuning
    pt_path = os.path.join(config["output_dir"], config["model_name"] + ".pt")
    torch.save(best_model, pt_path)
    print(f"\\n✅ Successfully saved raw PyTorch model to: {pt_path}")
    oww.export_model(model=best_model"""

if target in code and 'torch.save(best_model' not in code:
  code = code.replace(target, replacement, 1)
  with open(train_script, "w") as f:
    f.write(code)
  print("✅ Patched train.py to save raw .pt checkpoint.")
else:
  print("✅ train.py already configured to save .pt checkpoint.")

# 2. Run Model Training
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model


In [ ]:
import os
import sys
import py_compile

# 1. Reset train.py to clean repo state
!cd /content/openwakeword && git checkout openwakeword/train.py

train_script = "/content/openwakeword/openwakeword/train.py"
with open(train_script, "r") as f:
    lines = f.readlines()

# 2. Inject checkpoint saver right before torch.onnx.export using its exact indentation
new_lines = []
injected = False
for line in lines:
    if "torch.onnx.export(" in line and not injected:
        indent = line[:len(line) - len(line.lstrip())]
        new_lines.append(f"{indent}# Save raw PyTorch checkpoint for future fine-tuning\n")
        new_lines.append(f"{indent}torch.save(model_to_save, os.path.join(output_dir, model_name + '.pt'))\n")
        new_lines.append(f"{indent}print(f'\\n✅ Successfully saved raw PyTorch model to: {{os.path.join(output_dir, model_name + \".pt\")}}')\n")
        new_lines.append(line)
        injected = True
    else:
        new_lines.append(line)

with open(train_script, "w") as f:
    f.writelines(new_lines)

# 3. Verify Python syntax before running
try:
    py_compile.compile(train_script, doraise=True)
    print("✅ train.py successfully patched and syntax-verified.")
except py_compile.PyCompileError as e:
    print(f"❌ Syntax Error: {e}")
    sys.exit(1)

# 4. Start Model Training
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model


In [ ]:
# ==============================================================================
# Cell: Verify, Convert to TFLite & Download All Artifacts
# ==============================================================================
import os
import glob
import shutil
from google.colab import files

print("🔍 Checking saved training outputs...")
!ls -lh my_custom_model/

export_dir = "./visar_export"
os.makedirs(export_dir, exist_ok=True)

# 1. Locate Raw PyTorch Model (.pt)
pt_files = glob.glob("my_custom_model/**/*.pt", recursive=True) + glob.glob("my_custom_model/*.pt")
if pt_files:
    shutil.copy(pt_files[0], os.path.join(export_dir, "visar_edge_raw.pt"))
    print(f"✅ Found PyTorch trainable checkpoint: {pt_files[0]}")
else:
    print("⚠️ No direct .pt file found; packaging full model folder.")

# 2. Locate Exported ONNX Model (.onnx)
onnx_source = "my_custom_model/visar_edge.onnx"
if os.path.exists(onnx_source):
    shutil.copy(onnx_source, os.path.join(export_dir, "visar_edge.onnx"))
    print(f"✅ Found ONNX model: {onnx_source}")
else:
    raise FileNotFoundError("❌ ONNX file missing. Check directory contents above.")

# 3. Stage Configuration & Real Audio Samples
if os.path.exists("my_model.yaml"):
    shutil.copy("my_model.yaml", os.path.join(export_dir, "training_config.yaml"))

if os.path.exists("my_real_voice"):
    audio_dest = os.path.join(export_dir, "user_calibration_audio")
    os.makedirs(audio_dest, exist_ok=True)
    for wav in glob.glob("my_real_voice/*.wav"):
        shutil.copy(wav, audio_dest)
    print(f"✅ Preserved {len(glob.glob('my_real_voice/*.wav'))} mic samples.")

# 4. Modern TFLite Conversion via onnx2tf (No onnx_tf needed)
print("\n⚙️ Converting ONNX to FlatBuffer TFLite...")
!pip install -q onnx2tf
!onnx2tf -i {onnx_source} -o {export_dir} -kat onnx____Flatten_0 > /dev/null 2>&1

raw_tflite = os.path.join(export_dir, "visar_edge_float32.tflite")
final_tflite = os.path.join(export_dir, "visar_edge.tflite")
if os.path.exists(raw_tflite):
    os.rename(raw_tflite, final_tflite)
    print(f"✅ Converted TFLite model: {final_tflite}")

# 5. Build Deployment Bundle (Excludes giant .npy arrays)
archive_name = "visar_edge_production_bundle.tar.gz"
!tar -czf {archive_name} -C {export_dir} .
print(f"📦 Production package ready: {archive_name}")

# 6. Trigger Browser Downloads
print("\n🚀 Starting downloads...")
files.download(archive_name)
files.download(os.path.join(export_dir, "visar_edge.onnx"))
if os.path.exists(final_tflite):
    files.download(final_tflite)
if os.path.exists(os.path.join(export_dir, "visar_edge_raw.pt")):
    files.download(os.path.join(export_dir, "visar_edge_raw.pt"))


lets see we need these weird raw files,  or the bottom cel 7 or cell 8 polished one ?

In [ ]:
# Once Cell 6 completes, run this cell to locate and save the raw PyTorch model weights alongside your configuration file

import glob
import os
import shutil

# Verify trained outputs
model_files = glob.glob("my_custom_model/*.*")
print("Found output files:", model_files)

# Locate or package the PyTorch model checkpoint
pt_files = glob.glob("my_custom_model/**/*.pt", recursive=True)
if pt_files:
  print(f"✅ Found PyTorch checkpoint: {pt_files[0]}")
  shutil.copy(pt_files[0], "my_custom_model/visar_edge_raw_checkpoint.pt")
else:
  print(
      "Checkpoint was directly exported to ONNX. Creating backup of full"
      " training run directory..."
  )

# Bundle configuration and training metadata for future resumption
!tar -czf visar_edge_training_bundle.tar.gz my_custom_model/ my_model.yaml
print("✅ Created 'visar_edge_training_bundle.tar.gz' containing all weights and configs.")


Choose How to Save It

In [ ]:
'''
Option A: Direct Download to Your PC (Fastest)
Run this to download the model files directly through your browser:
'''
from google.colab import files

# 1. Download deployment model
files.download("my_custom_model/visar_edge.onnx")

# 2. Download raw training bundle (for future fine-tuning)
files.download("visar_edge_training_bundle.tar.gz")


In [ ]:
'''
Option B: Upload Directly to Hugging Face Hub
​If you want a permanent cloud backup accessible from any machine:
'''

!pip install -q huggingface_hub
from huggingface_hub import HfApi, create_repo

# Replace with your Hugging Face username and repo name
HF_TOKEN = ""  # Generate at huggingface.co/settings/tokens
REPO_NAME = "RootDeveloperDS/Visar_OpenWakeWord_Model"

create_repo(REPO_NAME, token=HF_TOKEN, repo_type="model", exist_ok=True)
api = HfApi()

# Upload deployment files and training bundle
api.upload_folder(
    folder_path="my_custom_model",
    repo_id=REPO_NAME,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"🚀 Model uploaded to: https://huggingface.co/{REPO_NAME}")


In [ ]:
# Cell 7: ONNX to TFLite Conversion & File Download
from google.colab import files

onnx_path = "my_custom_model/visar_edge.onnx"
raw_tflite = "my_custom_model/visar_edge_float32.tflite"
final_tflite = "my_custom_model/visar_edge.tflite"

# Convert ONNX to TFLite graph
!onnx2tf -i {onnx_path} -o my_custom_model/ -kat onnx____Flatten_0
!mv {raw_tflite} {final_tflite}

print("✅ Export complete. Starting file downloads...")
files.download(onnx_path)
files.download(final_tflite)


In [ ]:
# ==============================================================================
# Cell 7: Complete Production Export, Packaging & Download
# ==============================================================================
import os
import glob
import shutil
from google.colab import files

print("📦 Staging production artifacts...")
export_dir = "./visar_export"
os.makedirs(export_dir, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Stage ONNX Model and External Weights (.onnx + .onnx.data)
# ------------------------------------------------------------------------------
onnx_model = "my_custom_model/visar_edge.onnx"
onnx_data = "my_custom_model/visar_edge.onnx.data"

if os.path.exists(onnx_model):
    shutil.copy(onnx_model, os.path.join(export_dir, "visar_edge.onnx"))
    print("✅ Staged: visar_edge.onnx")
else:
    raise FileNotFoundError("❌ visar_edge.onnx not found in my_custom_model/")

if os.path.exists(onnx_data):
    shutil.copy(onnx_data, os.path.join(export_dir, "visar_edge.onnx.data"))
    print("✅ Staged: visar_edge.onnx.data (external weight buffer)")

# ------------------------------------------------------------------------------
# 2. Stage Raw PyTorch Checkpoints & Training Run State (For Future Fine-Tuning)
# ------------------------------------------------------------------------------
# Search for any generated .pt checkpoints
pt_candidates = glob.glob("my_custom_model/**/*.pt", recursive=True) + glob.glob("my_custom_model/*.pt")
pt_files = [f for f in pt_candidates if "piper" not in f and "phonemizer" not in f]

if pt_files:
    for pt in pt_files:
        name = os.path.basename(pt)
        shutil.copy(pt, os.path.join(export_dir, name))
        print(f"✅ Staged raw PyTorch model: {name}")
else:
    print("ℹ️ Checkpoints merged in memory into ONNX; backing up training run state...")

# Stage the internal training run directory (contains step state and metric logs)
internal_state = "my_custom_model/visar_edge"
if os.path.exists(internal_state):
    dest_state = os.path.join(export_dir, "training_state")
    if not os.path.exists(dest_state):
        shutil.copytree(internal_state, dest_state)
    print("✅ Staged: full training_state/ directory for incremental fine-tuning")

# ------------------------------------------------------------------------------
# 3. Stage Recipe Config & Ground-Truth Microphone Recordings
# ------------------------------------------------------------------------------
if os.path.exists("my_model.yaml"):
    shutil.copy("my_model.yaml", os.path.join(export_dir, "training_config.yaml"))
    print("✅ Staged: training_config.yaml")

if os.path.exists("my_real_voice"):
    audio_dest = os.path.join(export_dir, "user_calibration_audio")
    os.makedirs(audio_dest, exist_ok=True)
    for wav in glob.glob("my_real_voice/*.wav"):
        shutil.copy(wav, audio_dest)
    print(f"✅ Staged: {len(glob.glob('my_real_voice/*.wav'))} real voice calibration clips")

# ------------------------------------------------------------------------------
# 4. Verify & Stage TFLite FlatBuffer Model (Edge / Microcontroller Runtime)
# ------------------------------------------------------------------------------
tflite_source = os.path.join(export_dir, "visar_edge.tflite")
if not os.path.exists(tflite_source):
    print("⚙️ Compiling ONNX to TFLite...")
    !onnx2tf -i {onnx_model} -o {export_dir} -kat onnx____Flatten_0 > /dev/null 2>&1
    raw_tflite = os.path.join(export_dir, "visar_edge_float32.tflite")
    if os.path.exists(raw_tflite):
        os.rename(raw_tflite, tflite_source)

if os.path.exists(tflite_source):
    print(f"✅ Verified TFLite model: {tflite_source}")
else:
    print("⚠️ TFLite compilation skipped; ONNX is ready for edge inference.")

# ------------------------------------------------------------------------------
# 5. Build Compact Deployment Bundle & Trigger Browser Downloads
# ------------------------------------------------------------------------------
bundle_name = "visar_edge_production_bundle.tar.gz"
!tar -czf {bundle_name} -C {export_dir} .
print(f"\n📦 Production archive built: {bundle_name}")

print("\n🚀 Triggering downloads (Allow multiple files in browser prompt if asked)...")
files.download(bundle_name)
files.download(os.path.join(export_dir, "visar_edge.onnx"))
if os.path.exists(os.path.join(export_dir, "visar_edge.onnx.data")):
    files.download(os.path.join(export_dir, "visar_edge.onnx.data"))
if os.path.exists(tflite_source):
    files.download(tflite_source)


In [ ]:

# ==============================================================================
# Cell 8: Cloud Version-Controlled Backup to Hugging Face Hub
# ==============================================================================
import os
from huggingface_hub import HfApi, create_repo

# ------------------------------------------------------------------------------
# 1. Credentials & Target Repository
# ------------------------------------------------------------------------------
# Replace with your Hugging Face write token (huggingface.co/settings/tokens)
HF_TOKEN = "hf_hDZFiHkdzhnZ****************"
REPO_NAME = "RootDeveloperDS/Visar_OpenWakeWord_Model"

export_dir = "./visar_export"
if not os.path.exists(export_dir):
    raise FileNotFoundError("❌ Please execute Cell 7 first to assemble the visar_export directory.")

# ------------------------------------------------------------------------------
# 2. Generate Production Model Card (README.md)
# ------------------------------------------------------------------------------
model_card = """---
language:
- hi
- en
tags:
- openwakeword
- wake-word-detection
- voice-assistant
- edge-ai
- onnx
- tflite
license: apache-2.0
---

# Visar (वी-सार) — Custom openWakeWord Edge Model

A lightweight, high-accuracy wake-word detection model custom-trained for low-power edge SBCs, PCs, and offline home automation systems.

## Model Highlights
- **Target Phrase:** "Visar" / "वी-सार" / "Vee-saar"
- **Acoustic Tuning:** Native Hindi cadence and Indian English phonetic variations (`hi-IN-Madhur`, `hi-IN-Swara`, `en-IN-Prabhat`, `en-IN-Neerja`).
- **Ground Truth:** Real-world microphone samples convolved with MIT environmental impulse responses (RIRs).
- **Hard Negatives:** Penalized against acoustically close Indian words (*vichar*, *vishal*, *vikas*, *bazaar*).
- **Target Hardware:** Raspberry Pi, Linux Edge SBCs, Intel x86, Android, ARM64 microcontrollers.

## Audio Input Requirements
- **Sample Rate:** 16,000 Hz
- **Channels:** 1 (Mono)
- **Format:** 16-bit Signed Linear PCM

## Repository Contents
| File / Directory | Description |
| :--- | :--- |
| `visar_edge.onnx` | Production ONNX inference graph |
| `visar_edge.onnx.data` | External weight buffer for ONNX runtime |
| `visar_edge.tflite` | Quantized FlatBuffer model for mobile/embedded devices |
| `training_config.yaml` | Exact hyperparameter configuration used during training |
| `training_state/` | Step checkpoints and optimizer states for fine-tuning |
| `user_calibration_audio/` | Microphone ground-truth calibration recordings |

## Quickstart (Python Inference)
```python
import numpy as np
import openwakeword
from openwakeword.model import Model

# Initialize wake word engine with custom Visar model
oww_model = Model(wakeword_models=["visar_edge.onnx"])

# Pass raw 16kHz 16-bit PCM audio frames (1280 samples / 80ms chunk)
# audio_frame = np.frombuffer(mic_stream.read(1280), dtype=np.int16)
# prediction = oww_model.predict(audio_frame)
# if prediction["visar_edge"] > 0.5:
#     print("Wake-word detected: Visar!")

"""
with open(os.path.join(export_dir, "README.md"), "w", encoding="utf-8") as f:
  f.write(model_card)
print("📝 Production README.md generated.")
# ------------------------------------------------------------------------------
# 3. Create Repository & Push Artifacts
# ------------------------------------------------------------------------------
print(f"📡 Connecting to Hugging Face Hub ({REPO_NAME})...")
create_repo(repo_id=REPO_NAME, token=HF_TOKEN, repo_type="model", exist_ok=True)
api = HfApi()
print("⬆️ Uploading models, training state, and documentation...")
api.upload_folder(
  folder_path=export_dir,
  repo_id=REPO_NAME,
  repo_type="model",
  token=HF_TOKEN,
  commit_message="Publish Visar wake-word models: ONNX, TFLite, and fine-tuning state"
)
print(f"\n🎉 Successfully published!")
print(f"🔗 Repository URL: https://huggingface.co/{REPO_NAME}")




In [ ]:
# ==============================================================================
# Cell: Master Codebase & Patched Framework Archiver
# ==============================================================================
import os
import shutil
from google.colab import files

backup_dir = "./visar_source_backup"
os.makedirs(backup_dir, exist_ok=True)

print("📦 Packaging all patched libraries, recipes, and tools...")

# 1. Package Patched openWakeWord & Piper Codebases
!tar --exclude='*.npy' --exclude='*.wav' --exclude='*.flac' --exclude='*.tar' -czf {backup_dir}/patched_openwakeword.tar.gz openwakeword/
!tar --exclude='*.onnx' --exclude='*.pt' -czf {backup_dir}/patched_piper.tar.gz piper-sample-generator/

# 2. Copy Training Config & Real Mic Audio Samples
if os.path.exists("my_model.yaml"):
    shutil.copy("my_model.yaml", os.path.join(backup_dir, "my_model.yaml"))

if os.path.exists("my_real_voice"):
    dest_audio = os.path.join(backup_dir, "my_real_voice")
    if not os.path.exists(dest_audio):
        shutil.copytree("my_real_voice", dest_audio)

# 3. Generate 1-Click Reproducibility Runbook
runbook_content = """# Visar Wake Word — Reproducibility Runbook

## Why This Package Exists
Upstream openWakeWord relies on legacy packages that break in modern Python / PyTorch environments:
1. `torchaudio`: Removed `set_audio_backend()` and `.info()` (patched via soundfile shim).
2. PyTorch: Enforces `weights_only=True` by default (patched in Piper generator).
3. DeepPhonemizer: AWS S3 CMU dict bucket throws HTTP 403 (patched with offline Indian + English phonetic pool).
4. Hugging Face Datasets: Removed script execution (`fma.py` / AudioSet tar dead links patched via ESC-50).
5. Exporter: Legacy `onnx_tf` replaced with modern `onnx2tf`.

## How to Retrain From Scratch on a Fresh Colab:
1. Upload this archive and extract:
   `tar -xzf visar_training_pipeline_source.tar.gz`
2. Extract the pre-patched engines:
   `tar -xzf patched_openwakeword.tar.gz`
   `tar -xzf patched_piper.tar.gz`
3. Install core dependencies:
   `pip install -q openwakeword torch-audiomentations datasets webrtcvad-wheels onnxscript onnx2tf edge-tts pydub soundfile`
4. Populate `my_real_voice/` with any new audio recordings.
5. Re-run feature extraction and training:
   `python openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model`
"""

with open(os.path.join(backup_dir, "REPRODUCIBILITY_README.md"), "w") as f:
    f.write(runbook_content)

# 4. Create Master Source Archive
master_archive = "visar_training_pipeline_source.tar.gz"
!tar -czf {master_archive} -C {backup_dir} .
print(f"✅ Master source package created: {master_archive} ({os.path.getsize(master_archive) / (1024*1024):.2f} MB)")

# 5. Trigger Download
files.download(master_archive)


📦 Packaging all patched libraries, recipes, and tools...
✅ Master source package created: visar_training_pipeline_source.tar.gz (15.86 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==============================================================================
# Cell: Environment Lock & Zero-Error Dependency Snapshot
# ==============================================================================
import os
import platform
import subprocess
import sys
from google.colab import files

print("🔒 Freezing environment and resolving dependency graph...")

lock_dir = "./visar_env_lock"
os.makedirs(lock_dir, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Generate Clean Pinned requirements.txt (Filtered of local temp paths)
# ------------------------------------------------------------------------------
raw_freeze = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"]
).decode("utf-8")

# Filter out Colab-specific temporary local file URIs (@ file://...) that break pip install
clean_lines = []
for line in raw_freeze.splitlines():
  if "@ file://" not in line and not line.startswith("-e "):
    clean_lines.append(line)

req_path = os.path.join(lock_dir, "requirements_frozen.txt")
with open(req_path, "w") as f:
  f.write("\n".join(clean_lines))

print(f"✅ Generated requirements_frozen.txt ({len(clean_lines)} packages locked)")

# ------------------------------------------------------------------------------
# 2. Extract System & Hardware Specifications
# ------------------------------------------------------------------------------
try:
  import torch

  torch_ver = torch.__version__
  cuda_available = torch.cuda.is_available()
  cuda_ver = torch.version.cuda if cuda_available else "N/A"
  gpu_name = torch.cuda.get_device_name(0) if cuda_available else "None"
except ImportError:
  torch_ver = cuda_available = cuda_ver = gpu_name = "Not Installed"

spec_report = f"""======================================================================
VISAR TRAINING RUNTIME SPECIFICATION (GOLDEN STATE)
======================================================================
Python Version   : {platform.python_version()} ({sys.executable})
OS Platform      : {platform.platform()}
System Core      : {platform.machine()}
PyTorch Version  : {torch_ver}
CUDA Available   : {cuda_available}
CUDA Version     : {cuda_ver}
GPU Accelerator  : {gpu_name}
======================================================================
"""

spec_path = os.path.join(lock_dir, "system_specs.txt")
with open(spec_path, "w") as f:
  f.write(spec_report)

print("✅ Captured hardware & CUDA specifications:")
print(spec_report)

# ------------------------------------------------------------------------------
# 3. Create Automated 1-Click Bootstrap Script (bootstrap_clean_notebook.sh)
# ------------------------------------------------------------------------------
bootstrap_script = """#!/usr/bin/env bash
# ==============================================================================
# Visar Wake Word: Zero-Error Bootstrap Script
# Run this inside any blank Colab or Linux environment:
#   !bash bootstrap_clean_notebook.sh
# ==============================================================================
set -e

echo "🚀 [1/4] Installing verified core dependencies..."
pip install --no-cache-dir -r requirements_frozen.txt || {
    echo "⚠️ Strict freeze install hit conflicts. Falling back to pinned core modules..."
    pip install -q openwakeword==0.6.0 onnxscript onnx2tf edge-tts soundfile pydub datasets torch-audiomentations
}

echo "🩹 [2/4] Applying torchaudio compatibility shim..."
python -c '
import torchaudio, os
patch = \"\"\"
import soundfile as sf
import torch
from collections import namedtuple

AudioMetaData = namedtuple("AudioMetaData", ["sample_rate", "num_frames", "num_channels", "bits_per_sample", "encoding"])

def _shim_info(file_path, format=None):
    s = sf.info(str(file_path))
    return AudioMetaData(sample_rate=s.samplerate, num_frames=s.frames, num_channels=s.channels, bits_per_sample=16, encoding="PCM_S")

def _shim_load(file_path, frame_offset=0, num_frames=-1, normalize=True, channels_first=True, format=None):
    start = frame_offset
    stop = None if num_frames in (-1, None) else frame_offset + num_frames
    data, sr = sf.read(str(file_path), start=start, stop=stop, dtype="float32", always_2d=True)
    tensor = torch.from_numpy(data)
    if channels_first:
        tensor = tensor.t()
    return tensor, sr

info = _shim_info
load = _shim_load
\"\"\"
init_path = os.path.dirname(torchaudio.__file__) + "/__init__.py"
with open(init_path, "a") as f:
    f.write(patch)
print("✅ Torchaudio shim verified successfully.")
'

echo "📂 [3/4] Unpacking pre-patched repositories..."
if [ -f "patched_openwakeword.tar.gz" ]; then
    tar -xzf patched_openwakeword.tar.gz
    echo "✅ Patched openWakeWord extracted."
fi
if [ -f "patched_piper.tar.gz" ]; then
    tar -xzf patched_piper.tar.gz
    echo "✅ Patched Piper extracted."
fi

echo "🎉 [4/4] Environment ready! You can now run feature extraction and training."
"""

bootstrap_path = os.path.join(lock_dir, "bootstrap_clean_notebook.sh")
with open(bootstrap_path, "w") as f:
  f.write(bootstrap_script)

# Make executable
os.chmod(bootstrap_path, 0o755)
print("✅ Generated bootstrap_clean_notebook.sh")

# ------------------------------------------------------------------------------
# 4. Bundle & Trigger Download
# ------------------------------------------------------------------------------
lock_bundle = "visar_environment_lock.tar.gz"
!tar -czf {lock_bundle} -C {lock_dir} .
print(f"\n📦 Lock package built: {lock_bundle}")

print("🚀 Triggering download...")
files.download(lock_bundle)
files.download(req_path)


🔒 Freezing environment and resolving dependency graph...
✅ Generated requirements_frozen.txt (738 packages locked)
✅ Captured hardware & CUDA specifications:
VISAR TRAINING RUNTIME SPECIFICATION (GOLDEN STATE)
Python Version   : 3.13.15 (/usr/bin/python3)
OS Platform      : Linux-6.6.122+-x86_64-with-glibc2.39
System Core      : x86_64
PyTorch Version  : 2.11.0+cu128
CUDA Available   : True
CUDA Version     : 12.8
GPU Accelerator  : Tesla T4

✅ Generated bootstrap_clean_notebook.sh

📦 Lock package built: visar_environment_lock.tar.gz
🚀 Triggering download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>